In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace, _events
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._events import EventLoggerProvider
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client
from agents_tools_helpers import selector_func

In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [4]:
# Set up resource
resource = Resource(attributes={
    "service.name": "autogen-test-agentchat"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint="http://localhost:4318/v1/traces"
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)

# Set up logger provider
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(
    BatchLogRecordProcessor(OTLPLogExporter(endpoint="http://localhost:4318/v1/logs"))
)
_events.set_event_logger_provider(EventLoggerProvider(logger_provider))

# Enable OpenAI instrumentation
OpenAIInstrumentor().instrument()

In [5]:
rag_vector_memory = await rag_indexer_runner('/run/media/jun/Jun 6T/Code/TFM_BA_GPT/lib/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [7]:
model_client = load_model_client("azure")

In [8]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY', 'azure_endpoint': 'https://224d2-m9kjdmy0-eastus2.openai.azure.com/', 'azure_deployment': 'gpt-4o', 'api_version': '2024-12-01-preview'}
create_args: {'model': 'gpt-4o'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config_user_proxy_agent.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    AGENTS_CONFIG,
    agent_names=agent_names,
    model_client=model_client,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [10]:
# 1️⃣ Grab the existing writer agent
old_writer = agents["functional_spec_writer_agent"]

# 2️⃣ Wrap the sub-agents as tools
use_case_tool = AgentTool(agent=agents["use_case_agent"])
glossary_tool = AgentTool(agent=agents["glossary_agent"])

# 3️⃣ Collect the writer's current tools (if any)
existing_tools = getattr(old_writer, "_tools", []) or []

# 4️⃣ Combine tools: existing ones + new agent-tools
combined_tools = existing_tools + [use_case_tool, glossary_tool]

# 5️⃣ Extract the system message text (AutoGen stores it as a list of SystemMessage objects)
system_message_text = ""
if hasattr(old_writer, "_system_messages") and old_writer._system_messages:
    system_message_text = old_writer._system_messages[0].content  # first SystemMessage content

# 6️⃣ Rebuild functional_spec_writer_agent with combined tools
agents["functional_spec_writer_agent"] = AssistantAgent(
    name=old_writer._name,
    description=old_writer._description,
    model_client=old_writer._model_client,
    memory=old_writer._memory,
    tools=combined_tools,
    reflect_on_tool_use=old_writer._reflect_on_tool_use,
    system_message=system_message_text,
    model_client_stream=old_writer._model_client_stream,
)

print("✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.")


✅ functional_spec_writer_agent rebuilt with use_case_tool and glossary_tool attached.


In [11]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))




--- Agent: planning_agent ---
{'_description': 'The orchestrator agent responsible for breaking down a new '
                 'task into subtasks, assigning them to the right agents, and '
                 'ensuring the workflow progresses smoothly until completion.',
 '_handoff_tools': [],
 '_handoffs': {},
 '_is_running': False,
 '_max_tool_iterations': 1,
 '_memory': None,
 '_metadata': {},
 '_model_client': <autogen_ext.models.openai._openai_client.AzureOpenAIChatCompletionClient object at 0x7f9ed40eaf90>,
 '_model_client_stream': True,
 '_model_context': <autogen_core.model_context._unbounded_chat_completion_context.UnboundedChatCompletionContext object at 0x7f9ed3fe8050>,
 '_name': 'planning_agent',
 '_output_content_type': None,
 '_output_content_type_format': None,
 '_reflect_on_tool_use': False,
 '_structured_message_factory': None,
 '_system_messages': [SystemMessage(content='You are the PLANNING AGENT, the coordinator for this multi-agent system.\n\n**YOUR MISSION**\n  Brea

In [12]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")



--- Agent: planning_agent ---
Tools:

--- Agent: interviewer_agent ---
Tools:
Tool Name: save_conversation_to_jsonl
Description: 
Saves either:
1) A Q&A pair (interviewer + user) in one atomic JSONL write, OR
2) A generic list of records.



--- Agent: summarizer_agent ---
Tools:

--- Agent: functional_spec_writer_agent ---
Tools:
Tool Name: write_md_file_tool
Description: 
Writes the functional specification to a Markdown file.


Tool Name: use_case_agent
Description: An agent dedicated to generating Use Cases and User Stories from the project information.

Tool Name: glossary_agent
Description: An agent dedicated to building a Glossary of terms from the gathered requirements and project information.


--- Agent: functional_spec_checker_agent ---
Tools:
Tool Name: read_md_file_tool
Description: 
Reads the entire content of a Markdown file and returns it as a string.

Args:
    file_path (str): Optional path to the Markdown spec (defaults to DEFAULT_MD_SPEC_PATH).

Returns:
    str: T

In [13]:
selector_prompt = """
Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [14]:
# Order agents (workflow order), user_proxy last
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [17]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    model_client_streaming=False,
    selector_func=selector_func,
)

In [18]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [19]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [20]:
try:
    await Console(team.run_stream(task=task))
except Exception as e:
    print(f"Error during team run_stream: {e}")

await runtime.stop()

try:
    await model_client.close()
except Exception as e:
    print(f"Error closing model_client: {e}")

/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
interviewer_agent: not started  
summarizer_agent: not started  
functional_spec_writer_agent: not started  
functional_spec_checker_agent: not started  
diagram_creator_agent: not started  
mermaid_code_reviewer_agent: not started  

Breaking down the task:  
1. Start by gathering product requirements and scoping through an interview with the user.  
2. Summarize all collected information.  
3. Write the functional specification document.  
4. Review the functional specification for accuracy and completeness.  
5. Create process flow diagrams based on the final specification.  
6. Review the Mermaid code for the diagrams.  

Assigning the first task:  
1 interviewer_agent : Start gathering requirements from the user for the new product.  

[ASSIGNMENT] agent=interv

/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: interviewer_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: interviewer_agent (attempt 1)
Selected speaker: interviewer_agent
DEBUG:autogen_agentchat:Selected speaker: interviewer_agent


---------- MemoryQueryEvent (interviewer_agent) ----------
[MemoryContent(content='and printing requirements templates.Demo Video 8. RequirementsWatch this video on YouTube\xa0The analyst can talk you through their steps, techniques, tools, and processes for project completion. Their process may look something like this:Meet with stakeholdersGather and document requirementsGet approvalMonitor progress18. What Requirements Gathering Tool or Software Do You Use for Requirements Analysis?Requiment is an automated tool that lets analysts generate documents, wireframes, and task', mime_type='MemoryMimeType.TEXT', metadata={'mime_type': 'MemoryMimeType.TEXT', 'chunk_index': 28, 'source': 'https://www.requiment.com/business-analyst-requirements-gathering-interview-questions-answers/', 'score': 0.5285867154598236, 'id': 'e5855e4e-36ea-4436-ad7c-a32f564a6f56'}), MemoryContent(content='• \nFunctional Requirements Specification \n• \n1. Document Information  \no 1.1. Document Approvers & Reviewer

id='8e261648-5739-4ae3-8632-7e236795a3be' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 8, 9, 20, 35, 33, 155498, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_EKDeLwVUdcCEEFWVVZLHSi1L', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Can you briefly describe the new product and its primary purpose?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20231011162620"}', name='save_conversation_to_jsonl')] type='ToolCallRequestEvent'
id='8e261648-5739-4ae3-8632-7e236795a3be' source='inter

---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='call_EKDeLwVUdcCEEFWVVZLHSi1L', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Can you briefly describe the new product and its primary purpose?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The new product is an AI dialer that calls customers to offer travel packages. If the customer accepts, the call transfers to a human. If declined, the AI attempts to convince the customer up to three additional times before ending the call","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20231011162620"}', name='save_conversation_to_jsonl')]


... (200 outputs left out)

id='5e9d8c07-786c-4718-aacf-676701f51d0a' source='mermaid_code_reviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 9, 20, 59, 41, 834798, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Mermaid diagram successfully written to ../conversation/process_diagram.mmd', name='write_mermaid_to_file_tool', call_id='call_ULZNs9eJLGH9HR0Igxgo8MVz', is_error=False)] type='ToolCallExecutionEvent'
id='5e9d8c07-786c-4718-aacf-676701f51d0a' source='mermaid_code_reviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 8, 9, 20, 59, 41, 834798, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Mermaid diagram successfully written to ../conversation/process_diagram.mmd', name='write_mermaid_to_file_tool', call_id='call_ULZNs9eJLGH9HR0Igxgo8MVz', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='5e9d8c07-786c-4718-aacf-676701f51d0a' source='mermaid_code_reviewer_agent' mode

---------- ToolCallExecutionEvent (mermaid_code_reviewer_agent) ----------
[FunctionExecutionResult(content='Mermaid diagram successfully written to ../conversation/process_diagram.mmd', name='write_mermaid_to_file_tool', call_id='call_ULZNs9eJLGH9HR0Igxgo8MVz', is_error=False)]
---------- ModelClientStreamingChunkEvent (mermaid_code_reviewer_agent) ----------
Diagram code complete  
**READY**


/run/media/jun/Jun 6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)
Model selected a valid name: planning_agent (attempt 1)
DEBUG:autogen_agentchat:Model selected a valid name: planning_agent (attempt 1)
Selected speaker: planning_agent
DEBUG:autogen_agentchat:Selected speaker: planning_agent


---------- ModelClientStreamingChunkEvent (planning_agent) ----------
All tasks across the agents have successfully been completed. Here's a summary of the work performed:

### Summary:
1. **Interviewer_agent**: Gathered user requirements and scoped the AI dialer project. Successfully identified functionalities, integrations, KPIs, constraints, and dependencies.
2. **Summarizer_agent**: Consolidated all gathered information into a coherent summary, which was reviewed and approved.
3. **Functional_spec_writer_agent**: Drafted the functional specification document, including the key sections.
4. **Functional_spec_checker_agent**: Reviewed the specification document for completeness and approved it.
5. **Diagram_creator_agent**: Created process flow diagrams to visually represent the AI dialer system.
6. **Mermaid_code_reviewer_agent**: Reviewed the code for the diagrams for clarity and correctness, confirming readiness.

### Result:
The AI dialer requirements, functional specifications, 